# Heart Disease Classifier – Inference & Model Evaluation

**MLOps Assignment 01 – BITS Pilani (AIMLCZG523)**

This notebook covers:
1. Loading the saved best model pipeline
2. Single-record inference (dict input)
3. Batch inference (DataFrame input)
4. Confidence score analysis
5. Full test-set evaluation (confusion matrix, ROC curve, classification report)
6. Feature importance visualisation
7. Model metadata inspection

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import json
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, classification_report,
)

from src.inference import load_model, predict
from src.data_processing import prepare_data, get_feature_columns

SCREENSHOTS = os.path.abspath(os.path.join('..', 'screenshots'))
os.makedirs(SCREENSHOTS, exist_ok=True)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
print('Setup complete')

Setup complete


## 1 – Load Saved Model Pipeline

In [2]:
MODEL_PATH = os.path.abspath('../models/best_model.joblib')
META_PATH  = os.path.abspath('../models/model_metadata.json')

pipeline = load_model(MODEL_PATH)
print(f'Model loaded from: {MODEL_PATH}')
print(f'Pipeline steps: {[(name, type(step).__name__) for name, step in pipeline.steps]}')

Model loaded from: /Users/I349236/IMPL/shwetha/bits/sem_3/Mlops/sunil/MLOps-Assignment-1/models/best_model.joblib
Pipeline steps: [('preprocessor', 'ColumnTransformer'), ('classifier', 'LogisticRegression')]


In [3]:
with open(META_PATH) as f:
    meta = json.load(f)

print('Model Metadata:')
print(json.dumps(meta, indent=2))

Model Metadata:
{
  "model_name": "LogisticRegression",
  "metrics": {
    "accuracy": 0.9211,
    "precision": 0.9394,
    "recall": 0.8857,
    "f1": 0.9118,
    "roc_auc": 0.9554
  },
  "num_features": [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak"
  ],
  "cat_features": [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal"
  ]
}


## 2 – Single-Record Inference (Dict Input)

In [4]:
# Sample patient record
sample_patient = {
    'age': 63, 'sex': 1, 'cp': 3, 'trestbps': 145, 'chol': 233,
    'fbs': 1, 'restecg': 0, 'thalach': 150, 'exang': 0,
    'oldpeak': 2.3, 'slope': 0, 'ca': 0, 'thal': 1,
}

result = predict(pipeline, sample_patient)

print('Input patient record:')
for k, v in sample_patient.items():
    print(f'  {k:10s}: {v}')
print()
print('Prediction result:')
print(json.dumps(result, indent=2))

Input patient record:
  age       : 63
  sex       : 1
  cp        : 3
  trestbps  : 145
  chol      : 233
  fbs       : 1
  restecg   : 0
  thalach   : 150
  exang     : 0
  oldpeak   : 2.3
  slope     : 0
  ca        : 0
  thal      : 1

Prediction result:
{
  "prediction": 0,
  "label": "No Heart Disease",
  "confidence": 0.6354,
  "probabilities": {
    "no_disease": 0.6354,
    "disease": 0.3646
  }
}


In [5]:
# Healthy patient example
healthy_patient = {
    'age': 45, 'sex': 0, 'cp': 0, 'trestbps': 110, 'chol': 180,
    'fbs': 0, 'restecg': 0, 'thalach': 175, 'exang': 0,
    'oldpeak': 0.0, 'slope': 2, 'ca': 0, 'thal': 2,
}

result_healthy = predict(pipeline, healthy_patient)
print('Healthy patient prediction:')
print(json.dumps(result_healthy, indent=2))

Healthy patient prediction:
{
  "prediction": 0,
  "label": "No Heart Disease",
  "confidence": 0.9032,
  "probabilities": {
    "no_disease": 0.9032,
    "disease": 0.0968
  }
}


## 3 – Batch Inference (DataFrame Input)

In [6]:
batch_records = [
    {'age': 63, 'sex': 1, 'cp': 3, 'trestbps': 145, 'chol': 233,
     'fbs': 1, 'restecg': 0, 'thalach': 150, 'exang': 0,
     'oldpeak': 2.3, 'slope': 0, 'ca': 0, 'thal': 1},
    {'age': 45, 'sex': 0, 'cp': 0, 'trestbps': 110, 'chol': 180,
     'fbs': 0, 'restecg': 0, 'thalach': 175, 'exang': 0,
     'oldpeak': 0.0, 'slope': 2, 'ca': 0, 'thal': 2},
    {'age': 55, 'sex': 1, 'cp': 2, 'trestbps': 130, 'chol': 250,
     'fbs': 0, 'restecg': 1, 'thalach': 160, 'exang': 1,
     'oldpeak': 1.5, 'slope': 1, 'ca': 1, 'thal': 2},
    {'age': 70, 'sex': 1, 'cp': 3, 'trestbps': 160, 'chol': 286,
     'fbs': 1, 'restecg': 2, 'thalach': 108, 'exang': 1,
     'oldpeak': 1.5, 'slope': 1, 'ca': 3, 'thal': 3},
]

batch_df = pd.DataFrame(batch_records)
batch_results = [predict(pipeline, row.to_dict()) for _, row in batch_df.iterrows()]

results_df = pd.DataFrame([
    {
        'age': r['age'], 'sex': r['sex'],
        'prediction': res['prediction'],
        'label': res['label'],
        'confidence': res['confidence'],
        'p_disease': res['probabilities']['disease'],
    }
    for r, res in zip(batch_records, batch_results)
])
print('Batch inference results:')
print(results_df.to_string(index=False))

Batch inference results:
 age  sex  prediction            label  confidence  p_disease
  63    1           0 No Heart Disease      0.6354     0.3646
  45    0           0 No Heart Disease      0.9032     0.0968
  55    1           1    Heart Disease      0.6701     0.6701
  70    1           1    Heart Disease      0.7448     0.7448


## 4 – Confidence Score Distribution on Test Set

In [7]:
DATA_PATH = os.path.abspath('../data/raw/heart_disease.csv')
X_train, X_test, y_train, y_test = prepare_data(DATA_PATH)

print(f'Test set size: {len(X_test)} samples')
print(f'Class distribution: {y_test.value_counts().to_dict()}')

Test set size: 76 samples
Class distribution: {0: 41, 1: 35}


In [8]:
y_prob = pipeline.predict_proba(X_test)[:, 1]
y_pred = pipeline.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confidence distribution by true label
prob_df = pd.DataFrame({'probability': y_prob, 'true_label': y_test.map({0: 'No Disease', 1: 'Disease'})})
for label, color in [('No Disease', '#2196F3'), ('Disease', '#F44336')]:
    subset = prob_df[prob_df['true_label'] == label]['probability']
    axes[0].hist(subset, bins=20, alpha=0.6, color=color, label=label, edgecolor='white')
axes[0].axvline(0.5, color='black', linestyle='--', lw=1.5, label='Decision boundary (0.5)')
axes[0].set_title('Predicted Probability Distribution by True Label')
axes[0].set_xlabel('P(Disease)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Correct vs incorrect predictions
correct = (y_pred == y_test)
axes[1].hist(y_prob[correct],   bins=20, alpha=0.6, color='green', label='Correct',   edgecolor='white')
axes[1].hist(y_prob[~correct],  bins=20, alpha=0.6, color='red',   label='Incorrect', edgecolor='white')
axes[1].axvline(0.5, color='black', linestyle='--', lw=1.5)
axes[1].set_title('Confidence: Correct vs Incorrect Predictions')
axes[1].set_xlabel('P(Disease)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.suptitle('Model Confidence Analysis – Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
out = os.path.join(SCREENSHOTS, 'inference_confidence_distribution.png')
fig.savefig(out, bbox_inches='tight')
plt.close(fig)
print('Saved:', out)

Saved: /Users/I349236/IMPL/shwetha/bits/sem_3/Mlops/sunil/MLOps-Assignment-1/screenshots/inference_confidence_distribution.png


## 5 – Full Test-Set Evaluation

In [9]:
metrics = {
    'Accuracy':  accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred, zero_division=0),
    'Recall':    recall_score(y_test, y_pred, zero_division=0),
    'F1':        f1_score(y_test, y_pred, zero_division=0),
    'ROC-AUC':   roc_auc_score(y_test, y_prob),
}

print('Test-Set Metrics:')
for k, v in metrics.items():
    print(f'  {k:12s}: {v:.4f}')

print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))

Test-Set Metrics:
  Accuracy    : 0.9211
  Precision   : 0.9394
  Recall      : 0.8857
  F1          : 0.9118
  ROC-AUC     : 0.9554

Classification Report:
              precision    recall  f1-score   support

  No Disease       0.91      0.95      0.93        41
     Disease       0.94      0.89      0.91        35

    accuracy                           0.92        76
   macro avg       0.92      0.92      0.92        76
weighted avg       0.92      0.92      0.92        76



In [10]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
axes[0].set_title(f'Confusion Matrix\n(Model: {meta["model_name"]})')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC AUC = {auc:.4f}')
axes[1].plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'ROC Curve\n(Model: {meta["model_name"]})')
axes[1].legend(loc='lower right')

plt.tight_layout()
out = os.path.join(SCREENSHOTS, 'inference_confusion_roc.png')
fig.savefig(out, bbox_inches='tight')
plt.close(fig)
print('Saved:', out)

Saved: /Users/I349236/IMPL/shwetha/bits/sem_3/Mlops/sunil/MLOps-Assignment-1/screenshots/inference_confusion_roc.png


## 6 – Feature Importance

In [11]:
clf = pipeline.named_steps['classifier']
preprocessor = pipeline.named_steps['preprocessor']

try:
    feature_names = list(preprocessor.get_feature_names_out())
except Exception:
    feature_names = [f'f{i}' for i in range(100)]

if hasattr(clf, 'feature_importances_'):
    importances = clf.feature_importances_
    title = 'Feature Importances'
elif hasattr(clf, 'coef_'):
    importances = np.abs(clf.coef_[0])
    title = 'Feature Importances (|Coefficient|)'
else:
    importances = None

if importances is not None:
    top_n = min(15, len(feature_names))
    indices = np.argsort(importances)[-top_n:]

    fig, ax = plt.subplots(figsize=(9, 6))
    colors = ['#F44336' if importances[i] > np.median(importances) else '#2196F3' for i in indices]
    ax.barh(range(top_n), importances[indices], color=colors, edgecolor='white')
    ax.set_yticks(range(top_n))
    ax.set_yticklabels([feature_names[i] for i in indices], fontsize=9)
    ax.set_xlabel('Importance')
    ax.set_title(f'{title} – Top {top_n} Features\n(Model: {meta["model_name"]})',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    out = os.path.join(SCREENSHOTS, 'inference_feature_importance.png')
    fig.savefig(out, bbox_inches='tight')
    plt.close(fig)
    print('Saved:', out)
else:
    print('Feature importance not available for this model type.')

Saved: /Users/I349236/IMPL/shwetha/bits/sem_3/Mlops/sunil/MLOps-Assignment-1/screenshots/inference_feature_importance.png


## 7 – Threshold Analysis

In [12]:
thresholds = np.arange(0.1, 0.91, 0.05)
rows = []
for t in thresholds:
    y_t = (y_prob >= t).astype(int)
    rows.append({
        'Threshold': round(t, 2),
        'Accuracy':  round(accuracy_score(y_test, y_t), 4),
        'Precision': round(precision_score(y_test, y_t, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, y_t, zero_division=0), 4),
        'F1':        round(f1_score(y_test, y_t, zero_division=0), 4),
    })

thresh_df = pd.DataFrame(rows)
print('Threshold Sensitivity Analysis:')
print(thresh_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
for col, color in [('Precision','#2196F3'), ('Recall','#F44336'), ('F1','#4CAF50'), ('Accuracy','#FF9800')]:
    ax.plot(thresh_df['Threshold'], thresh_df[col], marker='o', ms=4, label=col, color=color)
ax.axvline(0.5, color='black', linestyle='--', lw=1, label='Default (0.5)')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Metric Scores vs Decision Threshold', fontweight='bold')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
out = os.path.join(SCREENSHOTS, 'inference_threshold_analysis.png')
fig.savefig(out, bbox_inches='tight')
plt.close(fig)
print('\nSaved:', out)

Threshold Sensitivity Analysis:
 Threshold  Accuracy  Precision  Recall     F1
      0.10    0.5526     0.5072  1.0000 0.6731
      0.15    0.6447     0.5645  1.0000 0.7216
      0.20    0.7105     0.6140  1.0000 0.7609
      0.25    0.7632     0.6667  0.9714 0.7907
      0.30    0.7895     0.7021  0.9429 0.8049
      0.35    0.8684     0.8049  0.9429 0.8684
      0.40    0.8816     0.8421  0.9143 0.8767
      0.45    0.9079     0.9118  0.8857 0.8986
      0.50    0.9211     0.9394  0.8857 0.9118
      0.55    0.9079     0.9375  0.8571 0.8955
      0.60    0.8816     0.9333  0.8000 0.8615
      0.65    0.8421     0.9259  0.7143 0.8065
      0.70    0.8289     0.9583  0.6571 0.7797
      0.75    0.8026     1.0000  0.5714 0.7273
      0.80    0.7895     1.0000  0.5429 0.7037
      0.85    0.7237     1.0000  0.4000 0.5714
      0.90    0.6579     1.0000  0.2571 0.4091



Saved: /Users/I349236/IMPL/shwetha/bits/sem_3/Mlops/sunil/MLOps-Assignment-1/screenshots/inference_threshold_analysis.png


## 8 – Inference Summary

| Metric | Value |
|--------|-------|
| Best Model | LogisticRegression |
| Accuracy | 0.9211 |
| Precision | 0.9394 |
| Recall | 0.8857 |
| F1 Score | 0.9118 |
| ROC-AUC | 0.9554 |

**Key observations:**
- The model correctly identifies ~92% of test patients.
- High precision (0.94) means when the model predicts disease, it is almost always correct.
- Recall of 0.89 means ~11% of actual disease cases are missed (false negatives).
- ROC-AUC of 0.9554 indicates excellent discrimination between classes.
- Threshold analysis shows F1 is maximised near the default 0.5 boundary.